# ReviewIQ — NLP Pipeline (Prototype: Netflix)

Prototyping the pipeline on **one app (Netflix)** with a 10k sample, then scaling.

Flow: embeddings → clustering → label clusters → sentiment → LLM summary.

Reads `../data/processed/master_clean_lang.parquet`. No language filter (see HANDOFF §7).

## Stage 1 — Embeddings

In [1]:
import pandas as pd
from pathlib import Path

# Read the language-tagged clean data
reviews = pd.read_parquet("../data/processed/master_clean_lang.parquet")

# Prototype on ONE app
netflix = reviews[reviews["app_name"] == "netflix"].copy()

# Embedding gate: keep reviews with content length >= 10 (decision #4).
# NOTE: NO language filter here — we keep all languages (HANDOFF §7).
netflix["char_len"] = netflix["content"].str.len()
netflix = netflix[netflix["char_len"] >= 10]
print("Netflix reviews after length gate:", len(netflix))

# For a FAST first pass, work on a random 10k sample.
# Later, set SAMPLE_SIZE = None to run the full Netflix set.
SAMPLE_SIZE = 10_000
if SAMPLE_SIZE:
    netflix = netflix.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    netflix = netflix.reset_index(drop=True)

print("Working set:", len(netflix))

Netflix reviews after length gate: 93064
Working set: 10000


In [2]:
from sentence_transformers import SentenceTransformer

# First run downloads ~470 MB, then it's cached on disk for next time.
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Model loaded. Vector size:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\nithi\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nithi\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Vector size: 384


C:\Users\nithi\AppData\Local\Temp\ipykernel_5940\1835242314.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Vector size:", model.get_sentence_embedding_dimension())


In [3]:
import numpy as np

texts = netflix["content"].tolist()

# Turn every review into a vector. The progress bar lets you watch it work.
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print("Embeddings shape:", embeddings.shape)   # (10000, 384)

# Save the vectors AND the matching reviews together, so they stay aligned.
out = Path("../data/processed/embeddings")
out.mkdir(parents=True, exist_ok=True)
np.save(out / "netflix_sample.npy", embeddings)
netflix.to_parquet(out / "netflix_sample.parquet", index=False)
print("Saved:", out)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (10000, 384)
Saved: ..\data\processed\embeddings


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

i = 0  # try changing this to any row number
sims = cosine_similarity(embeddings[i:i+1], embeddings)[0]
top = sims.argsort()[::-1][1:6]   # the 5 most similar reviews (skipping itself)

print("QUERY:", texts[i])
print("\nMost similar reviews:")
for j in top:
    print(f"  ({sims[j]:.2f})  {texts[j][:100]}")

QUERY: I like it because I can watch videos

Most similar reviews:
  (0.76)  I like this because it's full of amazing movies, shows, and more I love it
  (0.76)  i love it i can watch all my movies and shows on the go.
  (0.74)  I like it very much
  (0.74)  I love it. I can watch all of my shows and catch up on things
  (0.74)  I Like it very much
